# ARCH／GARCH（自回歸條件異質變異，1982／1986）

- **核心原理**：建模波動率隨時間變化
- **變體**：EGARCH（log 變異）、GJR-GARCH（壞消息影響比較大：槓桿效果）；Student-t 誤差
- **推薦資料集**：S&P 500 日報酬率（1999–2018，約 5,000 個交易日）
- **實作**：✋ 手刻（變異數遞迴 + 最大概似 + `scipy.optimize`）；🧰 `arch` 套件對照
- **文獻**：Engle (1982)（ARCH，2003 年諾貝爾經濟學獎）；Bollerslev (1986)（GARCH）；Glosten, Jagannathan & Runkle (1993)（GJR）

本章依照 README 的「每章標準流程」：EDA → Baseline → 手寫模型 → 套件實作 → 調參 → 測試集評估 → 診斷 → 本章小結。

和第 1～3 章最大的不同：**預測的不是平均，而是變異數。**

| | 第 1～3 章 | 本章 GARCH |
| --- | --- | --- |
| 預測的對象 | $E[y_t \mid \text{過去}]$（水準、成長率） | $\text{Var}(r_t \mid \text{過去}) = \sigma_t^2$（**波動率**） |
| 日報酬率的平均 | — | 幾乎無法預測（效率市場）：自我相關 ≈ 0 |
| 日報酬率的「大小」 | — | **可以預測**：大的變動後面跟著大的變動（波動群聚，volatility clustering） |
| 怎麼評估 | MAE、RMSE、MASE | 真正的 $\sigma_t^2$ 看不到 → 用 $r_t^2$ 當代理，**QLIKE** 損失；**VaR**（風險值）的突破次數 |
| 估計 | OLS、最小化 SSE | **最大概似**（數值最佳化，有限制條件） |

一句話：**GARCH 說「今天的變異數 = 長期平均 + 昨天的意外（$\varepsilon^2$）的一部分 + 昨天的變異數的一部分」—— 又是一個指數平滑（第 1 章），只是平滑的對象是 $\varepsilon^2$。**

### 閱讀指南：本章的符號與跨章引用

下一格是「前置複習」；第 2、3A–3D、4、5 節開頭另有各自的符號表或名詞表。

**資料**：`load_sp500_returns()` 回傳 S&P 500 的日報酬率（`pd.Series`，日期索引）。本章一律換成**百分比**（1% = 1.0），因為 GARCH 的參數在這個尺度比較好估（數值不會太小）。train：1999–2016；test：2017–2018（2018 年 2 月、10–12 月有兩次大波動）。

**本章最常出現的符號**（各小節開頭有完整符號表）：

| 符號 | 意義 | 本章的值 |
| --- | --- | --- |
| $r_t$ | 第 $t$ 天的報酬率（%） | |
| $\mu$ | 平均報酬（常數） | 約 0.03～0.05 %／天 |
| $\varepsilon_t = r_t - \mu$ | 意外（innovation） | |
| $\sigma_t^2$ | **條件變異數**：用到第 $t-1$ 天的資訊，對第 $t$ 天的變異數的預測 | |
| $\omega, \alpha, \beta$ | GARCH(1,1) 的參數：常數、ARCH 項（昨天的意外）、GARCH 項（昨天的變異數） | |
| $\alpha + \beta$ | **持續性**（persistence）：越接近 1，衝擊消退得越慢 | |
| $\bar\sigma^2 = \omega / (1 - \alpha - \beta)$ | 長期（無條件）變異數 | |
| $z_t = \varepsilon_t / \sigma_t$ | 標準化殘差（模型對的話應該 ≈ 獨立、變異 1） | |
| VaR$_{q}$ | $q$ 風險值：「明天的報酬有 $q$ 的機率低於這個值」 | $q$ = 1%、5% |

**跨章引用對照**：

| 文中寫法 | 指的是 | 那裡的結論（重述） |
| --- | --- | --- |
| 第 1 章 | 指數平滑 | $\ell_t = \alpha y_t + (1-\alpha)\ell_{t-1}$：權重 $\alpha(1-\alpha)^k$ 指數遞減；RiskMetrics 的 EWMA 波動率就是對 $r^2$ 做 SES（$\lambda = 1 - \alpha = 0.94$）。 |
| 第 2 章 | ARIMA | AR(1)：$z_t = c + \phi z_{t-1} + \varepsilon_t$，平穩條件 $|\phi| < 1$，往長期平均 $c/(1-\phi)$ 收斂、速度由 $\phi$ 決定。GARCH(1,1) 是「$\varepsilon^2$ 的 ARMA(1,1)」，持續性 $\alpha + \beta$ 扮演 $\phi$ 的角色。 |
| 第一部分第 3 章 | 邏輯回歸 | 最大概似：最小化負 log-likelihood；沒有閉式解時用數值最佳化。 |
| 第一部分第 15 章 | GMM | 常態分布的 log 密度 $-\frac12[\log 2\pi + \log\sigma^2 + (x-\mu)^2/\sigma^2]$。 |

### 前置複習：本章會用到的舊知識

#### 數學

**1. 報酬率**：$r_t = 100(P_t / P_{t-1} - 1)$（%）。例：100 → 101 → 1%。

**2. 變異數與波動率**：$\sigma^2$ 的單位是 %²、$\sigma$ 是 %。年化波動率 $\approx \sigma_{\text{日}}\sqrt{252}$（一年約 252 個交易日）。例：日 1% → 年化約 15.9%。

**3. 厚尾與峰度**：常態分布的超額峰度（excess kurtosis）= 0；超額峰度 > 0 → 極端值比常態多（厚尾）。例：Student-t（自由度 $\nu$）的超額峰度 $= 6/(\nu - 4)$（$\nu > 4$）。

**4. 常態的負 log-likelihood**：一個點 $\varepsilon$、變異 $\sigma^2$：$\frac12[\log 2\pi + \log\sigma^2 + \varepsilon^2/\sigma^2]$。所有點加總 → NLL。

**5. 等比收斂**：$x_h = \bar x + \phi^{h-1}(x_1 - \bar x)$：從 $x_1$ 往長期平均 $\bar x$ 收斂，每一步差距乘 $\phi$。**半衰期** $= \log 0.5 / \log\phi$。例：$\phi = 0.99$ → 69 步。

**6. 分位數**：常態分布的 1% 分位數 $= -2.326$、5% = $-1.645$。VaR$_{1\%}$ $= \mu + \sigma_t \times (-2.326)$。

**7. 二項分布與概似比檢定**：$n$ 天、每天突破 VaR 的機率 $p$ → 突破次數 $x \sim \text{Binomial}(n, p)$，期望 $np$。Kupiec 檢定：$LR = -2\log\frac{(1-p)^{n-x}p^x}{(1-\hat p)^{n-x}\hat p^x}$（$\hat p = x/n$），在「突破率 = $p$」的虛無假設下 $\sim \chi^2_1$。

#### 程式

**8. `scipy.optimize.minimize(f, x0, method="SLSQP", bounds=..., constraints=[{"type": "ineq", "fun": g}])`**：$g(\theta) \ge 0$ 的不等式限制（例如 $1 - \alpha - \beta > 0$）。
**9. `pd.Series.rolling(window).var()`**：滾動變異數。**10. `scipy.stats.norm.ppf(q)`、`scipy.stats.chi2.sf(x, df)`**：分位數、右尾機率。

---
## 1. 資料與探索性資料分析（EDA）

GARCH 的三個「典型事實」（stylized facts）：報酬率本身幾乎沒有自我相關、**報酬率的平方（或絕對值）有很強的自我相關**、**厚尾**。

In [ ]:
import warnings

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
from learning_map.data import load_sp500_returns

warnings.filterwarnings("ignore", category=FutureWarning)
r_raw = load_sp500_returns().dropna()
r_all = (100 * r_raw if r_raw.abs().max() < 1 else r_raw).astype(float).rename("return_pct")  # 換成 %（若已經是 % 就不動）
r_train, r_test = r_all[:"2016"], r_all["2017":]
print("全部：", len(r_all), r_all.index[0].date(), "～", r_all.index[-1].date(), "；train：", len(r_train), "；test：", len(r_test))

fig, axes = plt.subplots(2, 1, figsize=(15, 6), sharex=True)
r_all.plot(ax=axes[0], lw=0.5)
axes[0].set_title("daily return (%)")
r_all.abs().rolling(21).mean().mul(np.sqrt(252) * np.sqrt(np.pi / 2)).plot(ax=axes[1], color="tab:red")
axes[1].set_title("rough annualized volatility (%): 21-day mean |r| × √(π/2) × √252")
for ax in axes:
    ax.axvspan(r_test.index[0], r_test.index[-1], color="gray", alpha=0.15)
plt.tight_layout()
plt.show()

In [ ]:
# 典型事實：自我相關、厚尾
# 觀察：
#   - r 的 lag 1～10 自我相關有多大？r²（或 |r|）呢？哪一個在 lag 50 還明顯 > 0？
#   - 超額峰度是多少？比常態多出多少「極端日」（|z| > 4）？
def acf(x: np.ndarray, max_lag: int) -> np.ndarray:
    """第 1 章的樣本自我相關 r_k。"""
    x = np.asarray(x, dtype=float) - np.mean(x)
    return np.array([(x[k:] * x[: len(x) - k]).sum() / (x**2).sum() for k in range(max_lag + 1)])


x_tr = r_train.to_numpy()
fig, axes = plt.subplots(1, 3, figsize=(18, 3.6))
for ax, (name, s) in zip(axes[:2], [("r", x_tr), ("r²", x_tr**2)]):
    ax.bar(np.arange(1, 61), acf(s, 60)[1:])
    ax.axhline(2 / np.sqrt(len(s)), color="gray", ls=":")
    ax.axhline(-2 / np.sqrt(len(s)), color="gray", ls=":")
    ax.set_title(f"ACF of {name} (train)")
z_std = (x_tr - x_tr.mean()) / x_tr.std()
stats.probplot(z_std, dist="norm", plot=axes[2])
axes[2].set_title("Q-Q plot vs normal")
plt.show()
print(f"r 的 ACF lag 1–3：{acf(x_tr, 3)[1:].round(3)}；r² 的 ACF lag 1、10、50：{acf(x_tr**2, 50)[[1, 10, 50]].round(3)}")
print(f"超額峰度 {stats.kurtosis(x_tr):.2f}、偏態 {stats.skew(x_tr):.2f}；|z| > 4 的天數 {int((np.abs(z_std) > 4).sum())}（常態下的期望 {len(z_std) * 2 * stats.norm.sf(4):.2f}）")

In [ ]:
# TODO 1（問答）EDA 小結
#
# (1) 報酬率本身的 ACF：lag 1 是多少？有實用的預測力嗎？（如果有，套利就能賺錢 —— 效率市場）
# (2) r² 的 ACF：lag 1、10、50 各多少？衰減得快還是慢？這代表「今天的波動大小」跟「一個月前、兩個月前的波動」有關 —— 第 2 章 AR 的 ACF 是怎麼衰減的？
# (3) 厚尾：超額峰度多少？|z| > 4 的天數 vs 常態的期望。如果用常態分布算 VaR，會高估還是低估極端損失的機率？
# (4) 波動率的圖：哪幾段特別高？（2000–2002 網路泡沫、2008 金融危機、2011 歐債、2015、2018）test 期間（灰色）的波動跟 train 比？
# 答：

---
## 2. Baseline 與評估方式

**評估**：每一天的**一步預測** $\hat\sigma_t^2$（只用到前一天為止的資料）。真正的 $\sigma_t^2$ 看不到，用 $r_t^2$（或 $\varepsilon_t^2$）當**有雜訊的代理**：

| 指標 | 定義 | 說明 |
| --- | --- | --- |
| **QLIKE** | $\frac1n\sum\Big[\frac{\varepsilon_t^2}{\hat\sigma_t^2} - \log\frac{\varepsilon_t^2}{\hat\sigma_t^2} - 1\Big]$ | 越小越好；對代理的雜訊穩健（Patton 2011），比 MSE 適合比較波動率預測 |
| MSE | $\frac1n\sum(\varepsilon_t^2 - \hat\sigma_t^2)^2$ | 被少數極端日主導 |
| 常態 log-likelihood | $-\frac1n\sum\frac12[\log 2\pi + \log\hat\sigma_t^2 + \varepsilon_t^2/\hat\sigma_t^2]$ | 越大越好（平均每天） |
| VaR 突破率 | $\frac1n\sum\mathbb 1[r_t < \hat\mu + \hat\sigma_t\,\Phi^{-1}(q)]$ | 應該接近 $q$（1%、5%）；Kupiec 檢定（TODO 7） |

**Baseline**（都只用過去的資料）：

| Baseline | $\hat\sigma_t^2$ |
| --- | --- |
| 常數 | train 的變異數 |
| 滾動 21 天／63 天 | 過去 21 天（約一個月）、63 天（一季）的樣本變異數 |
| **EWMA（RiskMetrics）** | $\hat\sigma_t^2 = \lambda\hat\sigma_{t-1}^2 + (1-\lambda)\varepsilon_{t-1}^2$，$\lambda = 0.94$（J.P. Morgan 1996 的業界標準） |

Baseline 沒有要調的參數，跟前幾章一樣直接在 test 上算。（$\varepsilon = r - \mu$，$\mu$ 用 train 的平均。）

In [ ]:
MU_TRAIN = float(x_tr.mean())
eps_all = (r_all - MU_TRAIN).to_numpy()
test_mask = (r_all.index >= r_test.index[0])


def vol_scores(sigma2: np.ndarray, eps: np.ndarray, r: np.ndarray, mu: float) -> dict:
    """sigma2：一步預測的條件變異數（跟 eps 對齊）。"""
    ratio = eps**2 / sigma2
    out = {
        "qlike": float(np.mean(ratio - np.log(ratio) - 1)),
        "mse": float(np.mean((eps**2 - sigma2) ** 2)),
        "loglik/day": float(-0.5 * np.mean(np.log(2 * np.pi) + np.log(sigma2) + ratio)),
    }
    for q in (0.01, 0.05):
        out[f"VaR{int(q * 100)}% hit"] = float(np.mean(r < mu + np.sqrt(sigma2) * stats.norm.ppf(q)))
    return out


def ewma_var(eps: np.ndarray, lam: float = 0.94, init: float | None = None) -> np.ndarray:
    """RiskMetrics：sigma2[t] 只用到 eps[t − 1]。"""
    s = np.empty(len(eps))
    s[0] = np.var(eps[:75]) if init is None else init
    for t in range(1, len(eps)):
        s[t] = lam * s[t - 1] + (1 - lam) * eps[t - 1] ** 2
    return s


r_np = r_all.to_numpy()
base_sigma2 = {
    "constant (train var)": np.full(len(r_all), x_tr.var()),
    "rolling 21d": pd.Series(eps_all).rolling(21).var().shift(1).to_numpy(),
    "rolling 63d": pd.Series(eps_all).rolling(63).var().shift(1).to_numpy(),
    "EWMA λ=0.94": ewma_var(eps_all),
}
test_table = pd.DataFrame({k: vol_scores(v[test_mask], eps_all[test_mask], r_np[test_mask], MU_TRAIN) for k, v in base_sigma2.items()}).T
display(test_table.round(4))

In [ ]:
# TODO 2（問答）Baseline
#
# (1) 哪一個 baseline 的 QLIKE 最低？常數變異數為什麼最差？
# (2) 滾動 21 天 vs 63 天：誰比較好？窗口太短、太長各有什麼問題？（第 1 章：α 大 vs α 小）
# (3) VaR 1% 的突破率：理想是 1%。哪個 baseline 最接近？常數變異數的突破集中在哪些日子？
# 答：

---
## 3. 手寫模型：GARCH(1,1)

| 小節 | 做什麼 | TODO |
| --- | --- | --- |
| 3A | 變異數遞迴 $\sigma_t^2$ | 3 |
| 3B | 常態的負 log-likelihood；用 `scipy.optimize` 做最大概似估計；對照 `arch`；TODO 3–5 的問答 | 4、6 |
| 3C | 多步的變異數預測 | 5 |
| 3D | VaR 的 Kupiec 檢定 | 7 |
| 3E | 實驗：持續性與半衰期、Student-t、新聞衝擊曲線 | 8 |

**模型**：

$$r_t = \mu + \varepsilon_t,\qquad \varepsilon_t = \sigma_t z_t,\ z_t \sim N(0, 1),\qquad \sigma_t^2 = \omega + \alpha\,\varepsilon_{t-1}^2 + \beta\,\sigma_{t-1}^2$$

限制：$\omega > 0$、$\alpha, \beta \ge 0$、$\alpha + \beta < 1$（平穩；長期變異數 $\bar\sigma^2 = \omega/(1 - \alpha - \beta)$ 才存在）。

### 3A. 變異數遞迴

第一天沒有「昨天」。跟 `arch` 一樣用 **backcast**：把「第 0 天的 $\varepsilon^2$ 與 $\sigma^2$」都設成 $\hat\sigma^2_{\text{back}}$ = 前 75 天 $(r - \bar r)^2$ 的指數加權平均（權重 $0.94^k$，`backcast()` 已寫好）。所以 $\sigma_1^2 = \omega + \alpha\hat\sigma^2_{\text{back}} + \beta\hat\sigma^2_{\text{back}}$。

#### 符號表（3A–3D）

| 符號 | 意義 | 程式裡 |
| --- | --- | --- |
| $\varepsilon$ | $r - \mu$ | `eps` |
| $\hat\sigma^2_{\text{back}}$ | backcast | `bc` |
| $\sigma^2$ | 條件變異數（跟 `eps` 對齊：`sigma2[t]` 用到 `eps[t − 1]`） | `sigma2` |
| `prev_e2`、`prev_s2` | 「昨天」的 $\varepsilon^2$、$\sigma^2$ | |

**下標公式**：
- TODO 3：迴圈裡 `sigma2[t] = omega + alpha * prev_e2 + beta * prev_s2`；接著 `prev_e2, prev_s2 = eps[t] ** 2, sigma2[t]`。

#### 最小例子：TODO 3、4

$\varepsilon = (1, -2, 0)$、$\omega = 0.1$、$\alpha = 0.2$、$\beta = 0.7$、backcast = 1。

1. **正確答案**（手算）：$\sigma_1^2 = 0.1 + 0.2 \cdot 1 + 0.7 \cdot 1 = 1.0$；$\sigma_2^2 = 0.1 + 0.2 \cdot 1^2 + 0.7 \cdot 1.0 = 1.0$；$\sigma_3^2 = 0.1 + 0.2 \cdot (-2)^2 + 0.7 \cdot 1.0 = 1.6$。
   NLL $= \frac12\sum[\log 2\pi + \log\sigma^2 + \varepsilon^2/\sigma^2] = \frac12[3 \times 1.8379 + \log 1.6 + (1 + 4 + 0)] \approx 5.4918$。
2. **積木**：第 2 天：「昨天的意外」是 $\varepsilon_1^2 = 1$，不是 $\varepsilon_2^2$。
3. **留給你**：TODO 3、4。

In [ ]:
# 最小例子：TODO 3、4 —— 1. 正確答案：迴圈
eps_mini = np.array([1.0, -2.0, 0.0])
s_mini, pe, ps = [], 1.0, 1.0
for e in eps_mini:
    v = 0.1 + 0.2 * pe + 0.7 * ps
    s_mini.append(v)
    pe, ps = e**2, v
print("σ² =", s_mini, "；NLL =", round(0.5 * sum(np.log(2 * np.pi) + np.log(v) + e**2 / v for e, v in zip(eps_mini, s_mini)), 4))
# 寫完 TODO 3、4、執行過下一格之後，取消下面的註解來對照（應為 [1. 1. 1.6] 與 5.4918）：
# print(garch_sigma2(eps_mini, 0.1, 0.2, 0.7, bc=1.0), round(gaussian_nll(eps_mini, np.array([1.0, 1.0, 1.6])), 4))

In [ ]:
from typing import Self

from scipy.optimize import minimize


def backcast(eps: np.ndarray) -> float:
    """同 arch：前 75 個 ε² 的指數加權平均（權重 0.94^k，正規化）。"""
    tau = min(75, len(eps))
    w = 0.94 ** np.arange(tau)
    return float(np.sum(eps[:tau] ** 2 * w / w.sum()))


def garch_sigma2(eps: np.ndarray, omega: float, alpha: float, beta: float, bc: float) -> np.ndarray:
    """GARCH(1,1) 的條件變異數 (n,)；sigma2[t] 用到 eps[t − 1]（第 0 天用 backcast）。"""
    sigma2 = np.empty(len(eps))
    prev_e2, prev_s2 = bc, bc
    for t in range(len(eps)):
        # TODO 3（程式）：見下標公式（兩行）
        raise NotImplementedError("TODO 3：變異數遞迴")
    return sigma2


def gaussian_nll(eps: np.ndarray, sigma2: np.ndarray) -> float:
    """常態的負 log-likelihood（總和）。"""
    # TODO 4（程式）：0.5 * Σ [log 2π + log σ² + ε² / σ²]
    raise NotImplementedError("TODO 4：負 log-likelihood")


class GARCHScratch:
    """r = μ + ε、GARCH(1,1)、常態誤差。屬性：params_ = {mu, omega, alpha, beta}、loglik_、sigma2_、bc_。"""

    def fit(self, r: np.ndarray) -> Self:
        r = np.asarray(r, dtype=float)
        self.r_ = r
        self.bc_ = backcast(r - r.mean())  # 同 arch：用「樣本平均」的殘差算 backcast，固定不動

        def nll(theta: np.ndarray) -> float:
            mu, omega, alpha, beta = theta
            eps = r - mu
            return gaussian_nll(eps, garch_sigma2(eps, omega, alpha, beta, self.bc_))

        v = r.var()
        x0 = [r.mean(), 0.05 * v, 0.1, 0.85]
        bounds = [(-10 * abs(r.mean()) - 1, 10 * abs(r.mean()) + 1), (1e-6 * v, 10 * v), (0.0, 1.0), (0.0, 1.0)]
        cons = [{"type": "ineq", "fun": lambda th: 1 - 1e-6 - th[2] - th[3]}]
        res = minimize(nll, x0, method="SLSQP", bounds=bounds, constraints=cons, options={"maxiter": 500, "ftol": 1e-10})
        self.params_ = dict(zip(["mu", "omega", "alpha", "beta"], map(float, res.x)))
        self.loglik_ = -float(res.fun)
        self.success_ = bool(res.success)
        self.sigma2_ = self.filter(r)
        return self

    def filter(self, r: np.ndarray) -> np.ndarray:
        """用估好的參數，對（可以更長的）r 算一步預測的 σ²（backcast 用估計時的那一個）。"""
        p = self.params_
        return garch_sigma2(np.asarray(r, dtype=float) - p["mu"], p["omega"], p["alpha"], p["beta"], self.bc_)

    def forecast_var(self, h: int) -> np.ndarray:
        """從資料的最後一天往後 h 天的變異數預測 σ²_{T+1..T+h}。"""
        p = self.params_
        eps_T = self.r_[-1] - p["mu"]
        s_next = p["omega"] + p["alpha"] * eps_T**2 + p["beta"] * self.sigma2_[-1]  # σ²_{T+1}
        persist = p["alpha"] + p["beta"]
        s_bar = p["omega"] / (1 - persist)
        # TODO 5（程式）：σ²_{T+k} = s_bar + persist^(k − 1) · (s_next − s_bar)，k = 1..h（前置複習 5）
        raise NotImplementedError("TODO 5：多步預測")

### 3B. 最大概似估計、對照 `arch`

$\hat\theta = \arg\min_\theta \text{NLL}(\theta)$，$\theta = (\mu, \omega, \alpha, \beta)$；`SLSQP` 處理範圍與 $\alpha + \beta < 1$ 的限制（前置複習 8）。

`arch_model(r, mean="Constant", vol="GARCH", p=1, q=1, dist="normal").fit()` 用同樣的模型、同樣的 backcast：

1. **給定 `arch` 估出的參數**，手寫的 $\sigma_t$ 與 log-likelihood 應該**完全一樣**；
2. **各自估計**：參數差異 ~1e-4、log-likelihood 差異 ~1e-5（最佳化器不同）。

**多步預測**（3C，TODO 5）：$E[\sigma^2_{T+k}] = \bar\sigma^2 + (\alpha + \beta)^{k-1}(\sigma^2_{T+1} - \bar\sigma^2)$ —— 往長期變異數收斂，速度由持續性決定（第 2 章 AR(1) 的預測也是這個形狀）。對照 `arch` 的 `forecast(horizon=h)`。

In [ ]:
from arch import arch_model

am_n = arch_model(r_train, mean="Constant", vol="GARCH", p=1, q=1, dist="normal").fit(disp="off")
p_arch = am_n.params.to_numpy()  # mu, omega, alpha[1], beta[1]
mine = GARCHScratch().fit(x_tr)
eps_tr = x_tr - p_arch[0]
s2_at_arch = garch_sigma2(eps_tr, *p_arch[1:], bc=backcast(x_tr - x_tr.mean()))
print("arch 的參數：", am_n.params.round(5).to_dict(), "；log-lik", round(am_n.loglikelihood, 4))
print("手寫的參數：", {k: round(v, 5) for k, v in mine.params_.items()}, "；log-lik", round(mine.loglik_, 4), "；收斂？", mine.success_)
print(f"✅ 用 arch 的參數：σ_t 最大差異 {np.abs(np.sqrt(s2_at_arch) - am_n.conditional_volatility.to_numpy()).max():.1e}、"
      f"log-lik 差 {abs(-gaussian_nll(eps_tr, s2_at_arch) - am_n.loglikelihood):.1e}")
print(f"✅ 各自估計：參數最大差異 {np.abs(np.array(list(mine.params_.values())) - p_arch).max():.1e}、log-lik 差 {abs(mine.loglik_ - am_n.loglikelihood):.1e}")
fc_mine = mine.forecast_var(20)
fc_arch = am_n.forecast(horizon=20, reindex=False).variance.to_numpy()[0]
print(f"✅ 20 步變異數預測差異（手寫參數 vs arch 參數，兩者參數略不同）：{np.abs(fc_mine - fc_arch).max():.1e}")

In [ ]:
# TODO 6（問答）TODO 3–5 與對照
#
# (1) 用 arch 的參數時，σ_t 與 log-lik 的差異是多少？各自估計時參數差多少？為什麼不完全一樣？
# (2) 讀參數：α、β 各多少？α + β（持續性）多少？長期變異數 ω/(1 − α − β) 換成年化波動率是多少 %？（前置複習 2）跟 train 的樣本標準差比？
# (3) α 小、β 大代表什麼？（提示：β 是「昨天的變異數」的權重 —— 對照第 1 章，1 − β 有點像 SES 的 α）
# 答：

### 3D. VaR 與 Kupiec 檢定

VaR$_q$：明天的報酬有 $q$ 的機率低於 $\hat\mu + \hat\sigma_t\Phi^{-1}(q)$。如果模型對，「突破」（$r_t$ 低於 VaR）的天數 $x$ 應該 $\approx nq$。

**Kupiec 檢定**（前置複習 7）：
$$LR = -2\Big[(n - x)\log(1 - q) + x\log q - (n - x)\log(1 - \hat p) - x\log\hat p\Big],\quad \hat p = x/n$$
p 值 $= P(\chi^2_1 > LR)$；p < 0.05 → 突破率跟 $q$ 顯著不同（模型低估或高估風險）。$x = 0$ 時 $x\log\hat p$ 那項當作 0。

#### 最小例子：TODO 7

$n = 500$ 天、$q = 1\%$、突破 $x = 10$ 次（期望 5 次）→ $LR \approx 3.914$、p 值 $\approx 0.048$（剛好在 5% 的邊緣）。

In [ ]:
def kupiec(hits: np.ndarray, q: float) -> dict:
    """hits：每一天有沒有突破（0/1）。回傳 {"n", "x", "rate", "LR", "p_value"}。"""
    n, x = len(hits), int(np.sum(hits))
    p_hat = x / n
    # TODO 7（程式）：LR（x = 0 或 x = n 時，對應的 log 項當作 0）；p_value = stats.chi2.sf(LR, 1)
    raise NotImplementedError("TODO 7：Kupiec 檢定")
    return {"n": n, "x": x, "rate": p_hat, "LR": LR, "p_value": p_value}


# 寫完 TODO 7 之後，取消下面的註解來對照（應為 LR ≈ 3.9136、p ≈ 0.0479）：
# print(kupiec(np.r_[np.ones(10), np.zeros(490)], 0.01))

### 3E. 實驗

| 實驗 | 問題 |
| --- | --- |
| 1 | 條件波動率長什麼樣子？持續性、半衰期；多步預測怎麼收斂 |
| 2 | 標準化殘差 $z_t = \varepsilon_t / \sigma_t$：還有厚尾嗎？$z^2$ 還有自我相關嗎？ |
| 3 | 新聞衝擊曲線：昨天的 $\varepsilon$ 對今天的 $\sigma^2$ 的影響 —— GARCH（對稱）vs GJR-GARCH（壞消息加碼） |

In [ ]:
# 實驗 1：條件波動率、半衰期、多步預測
p = mine.params_
persist = p["alpha"] + p["beta"]
s_bar = p["omega"] / (1 - persist)
print(f"持續性 α + β = {persist:.4f}；半衰期 = {np.log(0.5) / np.log(persist):.0f} 天；長期年化波動率 = {np.sqrt(s_bar * 252):.1f}%")
fig, axes = plt.subplots(1, 2, figsize=(17, 3.8), gridspec_kw={"width_ratios": [3, 1]})
axes[0].plot(r_train.index, np.sqrt(mine.sigma2_ * 252), lw=0.8, label="GARCH σ_t (annualized %)")
axes[0].plot(r_train.index, np.sqrt(base_sigma2["EWMA λ=0.94"][: len(x_tr)] * 252), lw=0.6, alpha=0.7, label="EWMA")
axes[0].legend()
for start_date in ["2008-10-15", "2006-06-01"]:
    t0 = int(np.searchsorted(r_train.index, pd.Timestamp(start_date)))
    sub = GARCHScratch()
    sub.params_, sub.bc_, sub.r_ = mine.params_, mine.bc_, x_tr[: t0 + 1]
    sub.sigma2_ = mine.sigma2_[: t0 + 1]
    axes[1].plot(np.arange(1, 251), np.sqrt(sub.forecast_var(250) * 252), label=f"from {start_date}")
axes[1].axhline(np.sqrt(s_bar * 252), color="gray", ls=":", label="long-run")
axes[1].set_xlabel("days ahead")
axes[1].legend(fontsize=8)
axes[1].set_title("variance forecast → long-run")
plt.show()

In [ ]:
# 實驗 2：標準化殘差
z = (x_tr - p["mu"]) / np.sqrt(mine.sigma2_)
print(f"z 的標準差 {z.std():.3f}、超額峰度 {stats.kurtosis(z):.2f}（原始報酬 {stats.kurtosis(x_tr):.2f}）、偏態 {stats.skew(z):.2f}")
print("z² 的 ACF lag 1、10、50：", acf(z**2, 50)[[1, 10, 50]].round(3), "（原始 r²：", acf(x_tr**2, 50)[[1, 10, 50]].round(3), "）")
fig, ax = plt.subplots(figsize=(5, 4))
stats.probplot(z, dist="norm", plot=ax)
ax.set_title("standardized residuals vs normal")
plt.show()

In [ ]:
# 實驗 3：新聞衝擊曲線（σ²_{t−1} 固定在長期變異數）
am_gjr = arch_model(r_train, mean="Constant", vol="GARCH", p=1, o=1, q=1, dist="normal").fit(disp="off")
pg = am_gjr.params
e_grid = np.linspace(-5, 5, 201)
nic_garch = p["omega"] + p["alpha"] * e_grid**2 + p["beta"] * s_bar
nic_gjr = pg["omega"] + (pg["alpha[1]"] + pg["gamma[1]"] * (e_grid < 0)) * e_grid**2 + pg["beta[1]"] * s_bar
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(e_grid, nic_garch, label="GARCH(1,1)")
ax.plot(e_grid, nic_gjr, label="GJR-GARCH(1,1,1)")
ax.set_xlabel("yesterday's shock ε_{t−1} (%)")
ax.set_ylabel("today's σ²_t")
ax.legend()
plt.show()
print("GJR 的參數：", pg.round(4).to_dict(), f"；壞消息的 ARCH 係數 α + γ = {pg['alpha[1]'] + pg['gamma[1]']:.3f}、好消息 α = {pg['alpha[1]']:.3f}")

In [ ]:
# TODO 8（問答）實驗 1–3
#
# (1) 實驗 1：
#     (a) 持續性、半衰期、長期年化波動率各是多少？
#     (b) 從 2008-10-15（危機高點）開始的預測，要多少天才回到長期水準的一半距離？跟半衰期一致嗎？
#     (c) GARCH 與 EWMA 的曲線有什麼不同？（提示：EWMA 沒有 ω —— 它不會往「長期平均」收斂，λ = 0.94 的半衰期只有 11 天）
# 答：
#
# (2) 實驗 2：標準化之後，峰度降了多少？還是厚尾嗎？z² 的自我相關剩多少？這代表 GARCH 抓到了什麼、沒抓到什麼？
#     （提示：GARCH 解釋了「波動群聚」造成的那部分厚尾；剩下的厚尾 → 換 Student-t 誤差）
# 答：
#
# (3) 實驗 3：GJR 的 γ 是多少？同樣大小的「下跌 3%」和「上漲 3%」，隔天的 σ² 各是多少？這叫「槓桿效果」—— 為什麼股市會這樣？
#     GJR 裡 α 本身變成多少？（有時候會接近 0：好消息幾乎不增加波動）
# 答：

---
## 4. 套件實作：`arch` 的各種變體

| `arch_model(...)` 參數 | 模型 |
| --- | --- |
| `vol="GARCH", p=1, q=1, dist="normal"` | GARCH(1,1)、常態（= 手寫版） |
| `dist="t"` | Student-t 誤差（多估一個自由度 $\nu$：厚尾） |
| `vol="GARCH", p=1, o=1, q=1` | GJR-GARCH：$\sigma_t^2 = \omega + (\alpha + \gamma\mathbb 1[\varepsilon_{t-1} < 0])\varepsilon_{t-1}^2 + \beta\sigma_{t-1}^2$ |
| `vol="EGARCH", p=1, o=1, q=1` | EGARCH：對 $\log\sigma^2$ 建模（不需要正值限制）、也有不對稱項 |

在 train 上估計，比較 log-likelihood、AIC、BIC。

In [ ]:
variants = {
    "GARCH-N": dict(vol="GARCH", p=1, q=1, dist="normal"),
    "GARCH-t": dict(vol="GARCH", p=1, q=1, dist="t"),
    "GJR-N": dict(vol="GARCH", p=1, o=1, q=1, dist="normal"),
    "GJR-t": dict(vol="GARCH", p=1, o=1, q=1, dist="t"),
    "EGARCH-t": dict(vol="EGARCH", p=1, o=1, q=1, dist="t"),
}
fits = {name: arch_model(r_train, mean="Constant", **kw).fit(disp="off") for name, kw in variants.items()}
info = pd.DataFrame({name: {"loglik": f.loglikelihood, "AIC": f.aic, "BIC": f.bic, "# params": len(f.params)} for name, f in fits.items()}).T
display(info.round(2))
print("GJR-t 的參數：", fits["GJR-t"].params.round(4).to_dict())

In [ ]:
# TODO 9（問答）
#
# (1) AIC、BIC 誰最低？從 N 換成 t，log-likelihood 進步多少？加上不對稱項（GJR）呢？哪一個改進比較大？
# (2) GJR-t 的自由度 ν 是多少？對應的超額峰度 6/(ν − 4) 是多少？跟實驗 2 標準化殘差的峰度比？
# 答：

---
## 5. 調參：用 validation 期間選模型

時間序列的 hold-out：**1999–2014 估計參數、2015–2016 驗證**（一步預測的 QLIKE），選出最好的模型後，再用整個 train（1999–2016）重新估計，最後在 test（2017–2018）評估。

候選：EWMA（$\lambda$ = 0.94、0.97）、手寫 GARCH-N、`arch` 的 5 個變體。

**驗證期間的一步預測**：參數固定（只用 1999–2014 估的），把 2015–2016 的資料「餵進去」得到每一天的 $\sigma_t^2$。`arch` 用 `arch_model(更長的序列).fix(params)`；手寫版用 `filter()`。

In [ ]:
r_est, r_val = r_train[:"2014"], r_train["2015":]
r_upto_val = r_train
val_mask = r_upto_val.index >= r_val.index[0]
mu_est = float(r_est.mean())
eps_upto = (r_upto_val - mu_est).to_numpy()
val_rows = {}
for lam in (0.94, 0.97):
    s2 = ewma_var(eps_upto, lam)
    val_rows[f"EWMA λ={lam}"] = vol_scores(s2[val_mask], eps_upto[val_mask], r_upto_val.to_numpy()[val_mask], mu_est)
g_est = GARCHScratch().fit(r_est.to_numpy())
s2 = g_est.filter(r_upto_val.to_numpy())
val_rows["GARCH-N (scratch)"] = vol_scores(s2[val_mask], (r_upto_val.to_numpy() - g_est.params_["mu"])[val_mask], r_upto_val.to_numpy()[val_mask], g_est.params_["mu"])
for name, kw in variants.items():
    f_est = arch_model(r_est, mean="Constant", **kw).fit(disp="off")
    fixed = arch_model(r_upto_val, mean="Constant", **kw).fix(f_est.params)
    s2 = fixed.conditional_volatility.to_numpy() ** 2
    mu_f = float(f_est.params["mu"])
    val_rows[name] = vol_scores(s2[val_mask], (r_upto_val.to_numpy() - mu_f)[val_mask], r_upto_val.to_numpy()[val_mask], mu_f)
val_table = pd.DataFrame(val_rows).T.sort_values("qlike")
display(val_table.round(4))
chosen = val_table.index[0]
print("validation QLIKE 最低：", chosen)

In [ ]:
# TODO 10（問答）
#
# (1) validation 上的排名：GARCH 類比 EWMA 好多少？不對稱（GJR、EGARCH）有幫助嗎？t 分布呢？
#     （注意：QLIKE 只看變異數的預測，跟誤差分布是 N 還是 t 的關係比較間接；t 分布主要影響的是 log-likelihood 與 VaR）
# (2) 為什麼不直接用 AIC（第 4 節，train 上）選模型？AIC 和 validation QLIKE 選的一樣嗎？
# 答：

---
## 6. 測試集評估

選出的模型在整個 train（1999–2016）上重新估計，參數固定，對 test（2017–2018）的每一天做一步預測，只評估一次。也放進手寫 GARCH-N 與 EWMA 當參照點。VaR：常態的模型用 $\Phi^{-1}(q)$；t 的模型用（標準化）t 分布的分位數。成績寫進 `results/sp500_vol.csv`。

In [ ]:
from learning_map.results import compare, log_result

BENCH = "sp500_vol"
r_np_all = r_all.to_numpy()


def t_quantile(q: float, nu: float) -> float:
    """變異數為 1 的標準化 t 分布的分位數。"""
    return float(stats.t.ppf(q, nu) * np.sqrt((nu - 2) / nu))


final_rows = {}
# 手寫 GARCH-N（整個 train 估計：前面的 mine）
s2 = mine.filter(r_np_all)
final_rows["GARCH-N (scratch)"] = (s2, mine.params_["mu"], None)
final_rows["EWMA λ=0.94"] = (base_sigma2["EWMA λ=0.94"], MU_TRAIN, None)
if chosen.startswith("EWMA") and chosen not in final_rows:
    final_rows[chosen] = (ewma_var(eps_all, float(chosen.split("=")[1])), MU_TRAIN, None)
names_to_eval = list(dict.fromkeys([chosen, "GARCH-t", "GJR-t"]))
for name in names_to_eval:
    if name in variants:
        f_tr = fits[name]
        fixed = arch_model(r_all, mean="Constant", **variants[name]).fix(f_tr.params)
        nu = float(f_tr.params["nu"]) if "nu" in f_tr.params else None
        final_rows[name] = (fixed.conditional_volatility.to_numpy() ** 2, float(f_tr.params["mu"]), nu)

for name, (s2, mu_, nu) in final_rows.items():
    eps_ = r_np_all[test_mask] - mu_
    sc = vol_scores(s2[test_mask], eps_, r_np_all[test_mask], mu_)
    for q in (0.01, 0.05):
        qq = stats.norm.ppf(q) if nu is None else t_quantile(q, nu)
        hits = r_np_all[test_mask] < mu_ + np.sqrt(s2[test_mask]) * qq
        sc[f"VaR{int(q * 100)}% hit"] = float(hits.mean())
        sc[f"Kupiec p ({int(q * 100)}%)"] = kupiec(hits.astype(int), q)["p_value"]
    label = f"{name} (validation-selected)" if name == chosen else name
    test_table.loc[label] = sc
    log_result(BENCH, label, {k: round(v, 5) for k, v in sc.items()}, test="2017-2018")
display(test_table.round(4))
compare(BENCH, "qlike", higher_is_better=False)

---
## 7. 診斷：test 期間的波動與 VaR 突破

In [ ]:
best_s2, best_mu, best_nu = final_rows[chosen] if chosen in final_rows else final_rows["GARCH-N (scratch)"]
q01 = stats.norm.ppf(0.01) if best_nu is None else t_quantile(0.01, best_nu)
var01 = best_mu + np.sqrt(best_s2[test_mask]) * q01
r_t = r_np_all[test_mask]
hits = r_t < var01
fig, axes = plt.subplots(2, 1, figsize=(15, 7), sharex=True)
axes[0].plot(r_test.index, r_t, lw=0.6, color="k", label="return")
axes[0].plot(r_test.index, var01, color="tab:red", label=f"1% VaR ({chosen})")
axes[0].scatter(r_test.index[hits], r_t[hits], color="tab:red", zorder=3, label=f"breaches: {hits.sum()}")
axes[0].legend(fontsize=8)
axes[1].plot(r_test.index, np.sqrt(best_s2[test_mask] * 252), label=f"{chosen} σ (annualized)")
axes[1].plot(r_test.index, np.abs(r_t) * np.sqrt(252) * np.sqrt(np.pi / 2), lw=0.4, alpha=0.5, label="|r| × √(π/2) × √252 (noisy proxy)")
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()
print("突破的日期：", [d.strftime("%Y-%m-%d") for d in r_test.index[hits]])

In [ ]:
# TODO 11（問答，本章總結）
#
# (1) 測試集：依 QLIKE 排序。GARCH 類比 EWMA、常數變異數好多少？手寫 GARCH-N 跟 arch 的版本差多少？
# (2) VaR：
#     (a) 1%、5% 的突破率與 Kupiec p 值。哪些模型「通過」（p > 0.05）？常態 vs t 分布的 1% VaR 誰的突破率比較接近 1%？
#     (b) 突破集中在哪幾天（看圖）？「突破會群聚」違反了什麼假設？（提示：Christoffersen 的獨立性檢定）
# (3) 2018 年 2 月 5 日 S&P 500 跌了 4.1%：前一天模型預測的波動率是多少？隔天之後呢？GARCH「事前」能預測這種跳躍嗎？
# (4) 第 1～4 章總結：
#     |               | 指數平滑 | ARIMA | VAR | GARCH |
#     | 預測的對象    |          |       |     |       |
#     | 怎麼估計      |          |       |     |       |
#     | 怎麼評估      |          |       |     |       |
# 答：

---
## 8. 本章小結

（完成所有 TODO、並經過回饋與修正之後，依 README「章末小結」填寫：問答註解轉成 markdown、結果表、核心觀念、待補強。）

### 方法與結果

S&P 500 日報酬率（train 1999–2016、test 2017–2018；一步預測）

| 方法 | 做法 | 持續性 | validation QLIKE | test QLIKE | VaR 1% 突破率 | Kupiec p |
| --- | --- | --- | --- | --- | --- | --- |
| 常數變異數 | | — | — | | | |
| 滾動 21／63 天 | | — | — | | | |
| EWMA $\lambda = 0.94$ | | | | | | |
| GARCH-N（手寫） | | | | | | |
| GARCH-t、GJR-t、EGARCH-t（arch） | | | | | | |

### 核心觀念

1.
2.
3.

### 待補強（之後回來可以學）

**理論**
- ☐ GARCH(1,1) = $\varepsilon^2$ 的 ARMA(1,1)；四階動差存在的條件
- ☐ 擬最大概似（QMLE）：常態概似在誤差不是常態時仍一致；穩健標準誤（Bollerslev-Wooldridge）
- ☐ Christoffersen (1998) 的條件涵蓋檢定（獨立性 + 突破率）
- ☐ 預期損失（Expected Shortfall）與其回測

**延伸**
- ☐ 已實現波動率（realized volatility，日內資料）與 HAR 模型
- ☐ 多變量 GARCH（DCC）：資產之間的相關也會變
- ☐ 隨機波動率模型（stochastic volatility）；用 Kalman filter（第 5 章）或 MCMC 估計